In [1]:
import sys
from pathlib import Path

# Notebook nằm trong /Jupyter
ROOT = Path.cwd().parent               # project root: DL-DuDoan33MonAnVietNam
print("ROOT =", ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT = /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam


In [ ]:
# Cell 1: Import & config

import os
from pathlib import Path
import time
import copy
import json

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device)


In [ ]:
# Cell 2: Đường dẫn dữ liệu & transforms

DATA_DIR = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/Images")  # sửa cho đúng

train_dir = DATA_DIR / "Train"
val_dir   = DATA_DIR / "Validate"
test_dir  = DATA_DIR / "Test"

assert train_dir.exists(),  f"Không tìm thấy {train_dir}"
assert val_dir.exists(),    f"Không tìm thấy {val_dir}"
assert test_dir.exists(),   f"Không tìm thấy {test_dir}"

IMG_SIZE = 224  # EfficientNet-B0 chuẩn là 224x224

# chuẩn hóa theo ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = datasets.ImageFolder(train_dir, transform=train_tf)
val_set   = datasets.ImageFolder(val_dir,   transform=eval_tf)
test_set  = datasets.ImageFolder(test_dir,  transform=eval_tf)

class_names = train_set.classes
num_classes = len(class_names)

print("Số lớp:", num_classes)
print("Các lớp:", class_names[:10], "...")


In [ ]:
# Cell 3: DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 4  # chỉnh theo máy

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Train batches:", len(train_loader))
print("Val   batches:", len(val_loader))
print("Test  batches:", len(test_loader))


In [ ]:
# Cell 4: Định nghĩa EfficientNet-B0 fine-tune

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class MTL_EfficientNetB0(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # load weight ImageNet
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = self.backbone.classifier[1].in_features
        # thay classifier cuối
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = MTL_EfficientNetB0(num_classes=num_classes).to(device)
print(model.backbone.classifier)


In [ ]:
# Cell 5: Loss, optimizer, scheduler

import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

lr = 1e-3
weight_decay = 1e-4

# Loss
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr,
    weight_decay=weight_decay
)

# Scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",      # giảm khi loss giảm chậm
    factor=0.5,      # LR = LR * 0.5
    patience=3       # 3 epoch không cải thiện sẽ giảm LR
)

print("Optimizer & LR Scheduler đã sẵn sàng")


In [ ]:
# Cell 6: hàm train & eval

def run_one_epoch(model, loader, optimizer=None):
    """
    Nếu optimizer != None -> train, ngược lại -> eval
    """
    is_train = optimizer is not None
    model.train(is_train)

    epoch_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        epoch_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total   += labels.size(0)

    avg_loss = epoch_loss / total
    acc = correct / total
    return avg_loss, acc


In [ ]:
# Cell 7: Train loop

EPOCHS = 30
PATIENCE = 5   # early stopping

best_val_acc = 0.0
best_state = None
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

no_improve = 0
start_all = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = run_one_epoch(model, train_loader, optimizer)
    val_loss,   val_acc   = run_one_epoch(model, val_loader,   optimizer=None)

    scheduler.step(val_loss)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"[Epoch {epoch:02d}/{EPOCHS}] "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f} | "
          f"time={time.time()-t0:.1f}s")

    # lưu best model theo val_acc
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
        print("  👉 New best model, val_acc:", best_val_acc)
    else:
        no_improve += 1
        print("  😴 Không cải thiện, no_improve =", no_improve)

    if no_improve >= PATIENCE:
        print("⛔ Early stopping.")
        break

print("Train xong sau:", time.time() - start_all, "giây")


In [ ]:
# Cell 8: Lưu model & history

RUN_DIR = Path("runs") / f"mtl_efficientnet_b0_foods"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = RUN_DIR / "mtl_efficientnet_b0_best.pth"
HIS_PATH  = RUN_DIR / "history.json"

# lưu best_state
if best_state is not None:
    torch.save({"model_state_dict": best_state,
                "class_names": class_names},
               CKPT_PATH)
    print("✅ Đã lưu best checkpoint vào:", CKPT_PATH)

# lưu history
with open(HIS_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)
print("✅ Đã lưu history vào:", HIS_PATH)


In [ ]:
# Cell 9: Plot loss & acc

epochs = history["epoch"]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"],   label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Train acc")
plt.plot(epochs, history["val_acc"],   label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Confusion Matrix

cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))

plt.figure(figsize=(10, 10))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, class_names, rotation=90)
plt.yticks(tick_marks, class_names)

thresh = cm.max() / 2.
for i in range(num_classes):
    for j in range(num_classes):
        v = cm[i, j]
        if v > 0:
            plt.text(j, i, str(v),
                     horizontalalignment="center",
                     color="white" if v > thresh else "black",
                     fontsize=6)

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()


In [ ]:
# Cell 10: Load best checkpoint & evaluate trên Test

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

all_labels = []
all_preds  = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)

print("📌 Classification report (Test):")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))


In [ ]:
# Cell 2: Load YOLO model detect món ăn

# ĐƯỜNG DẪN MODEL YOLO ĐÃ TRAIN
CKPT_YOLO_PATH = Path(
    "/media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/runs_yolo/vnfoods_det/weights/best.pt"
)

if not CKPT_YOLO_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy YOLO checkpoint: {CKPT_YOLO_PATH}")

yolo_model = YOLO(str(CKPT_YOLO_PATH))
yolo_model.to(device)

print("✅ YOLO model đã load xong:", CKPT_YOLO_PATH.name)


In [ ]:
def detect_foods_yolo(pil_img: Image.Image,
                      conf_thres: float = 0.3,
                      iou_thres: float = 0.5):
    """
    Chạy YOLO trên 1 ảnh PIL, trả về:
      - boxes:  [N, 4]  (x1, y1, x2, y2)
      - scores: [N]     độ tin cậy YOLO
      - cls_ids:[N]     id class của YOLO (nếu có label)
    """
    # YOLO nhận numpy hoặc path đều được
    img_np = np.array(pil_img)

    results = yolo_model.predict(
        source=img_np,
        conf=conf_thres,
        iou=iou_thres,
        verbose=False
    )

    r = results[0]
    if r.boxes is None or len(r.boxes) == 0:
        return np.zeros((0, 4), dtype=int), np.array([]), np.array([])

    boxes = r.boxes.xyxy.cpu().numpy().astype(int)   # (N, 4)
    scores = r.boxes.conf.cpu().numpy()             # (N,)
    cls_ids = r.boxes.cls.cpu().numpy().astype(int) # (N,)

    print(f"Phát hiện {len(boxes)} vùng món ăn (YOLO).")
    return boxes, scores, cls_ids


In [ ]:
@torch.inference_mode()
def classify_yolo_regions(pil_img: Image.Image,
                          det_conf: float = 0.4,
                          det_iou: float = 0.5,
                          cls_thres: float = 0.5,
                          enlarge: float = 0.10):
    """
    - det_conf, det_iou: ngưỡng YOLO
    - cls_thres: ngưỡng tin cậy EfficientNet, dưới ngưỡng => UNKNOWN
    - enlarge: mở rộng bbox theo tỉ lệ (10% = 0.1)
    """
    w, h = pil_img.size

    boxes, det_scores, det_cls_ids = detect_foods_yolo(
        pil_img, conf_thres=det_conf, iou_thres=det_iou
    )

    if len(boxes) == 0:
        print("❗ YOLO không phát hiện vùng nào.")
        return [], pil_img

    draw_img = pil_img.copy()
    draw = ImageDraw.Draw(draw_img)

    results = []

    for i, (box, det_p) in enumerate(zip(boxes, det_scores)):
        x1, y1, x2, y2 = box

        # Mở rộng bbox 1 chút
        dx = int((x2 - x1) * enlarge)
        dy = int((y2 - y1) * enlarge)
        nx1 = max(0, x1 - dx)
        ny1 = max(0, y1 - dy)
        nx2 = min(w, x2 + dx)
        ny2 = min(h, y2 + dy)

        patch = pil_img.crop((nx1, ny1, nx2, ny2))

        # EfficientNet top-1
        cls_results = predict_image(patch, topk=1)
        top_name, top_prob = cls_results[0]

        if top_prob < cls_thres:
            label = f"UNKNOWN ({top_prob*100:.1f}%)"
            food_name = "UNKNOWN"
        else:
            label = f"{top_name} ({top_prob*100:.1f}%)"
            food_name = top_name

        # Vẽ khung
        draw.rectangle((nx1, ny1, nx2, ny2), outline="red", width=3)

        # Vị trí text
        text_x, text_y = nx1 + 4, ny1 + 4
        draw.text((text_x, text_y), label, fill="yellow")

        results.append({
            "box": (nx1, ny1, nx2, ny2),
            "yolo_score": float(det_p),
            "food_name": food_name,
            "cls_prob": float(top_prob),
        })

    print("Kết quả:")
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['food_name']:20s}  | cls={r['cls_prob']:.3f} | yolo={r['yolo_score']:.3f}")

    return results, draw_img


In [ ]:
# Cell 3: Hàm đếm số món ăn trên bàn (dùng lại classify_yolo_regions)

def detect_and_count_foods(
    pil_img,
    det_conf=0.4,       # ngưỡng YOLO (objectness)
    det_iou=0.5,        # IoU NMS
    cls_thres=0.5,      # ngưỡng tin cậy EfficientNet
    enlarge=0.12,       # nới rộng box một chút cho dễ nhìn
    verbose=True
):
    """
    - Nhận 1 ảnh PIL
    - Chạy YOLO để lấy các vùng nghi là món ăn
    - Dùng EfficientNet phân loại từng vùng
    - Trả ra:
        count      : số vùng được nhận diện là món ăn (không UNKNOWN)
        detections : list dict, mỗi dict là 1 món
        fig, ax    : figure matplotlib đã vẽ box + label
    """

    # Gọi lại hàm đã chạy OK ở cell trước
    results, vis_img = classify_yolo_regions(
        pil_img,
        det_conf=det_conf,
        det_iou=det_iou,
        cls_thres=cls_thres,
        enlarge=enlarge,
    )

    # results: list[{"label", "cls_name", "cls_conf", "yolo_conf", "box"}]
    # Lọc bỏ những vùng UNKNOWN nếu muốn
    detections = [r for r in results if r["label"] != "UNKNOWN"]

    count = len(detections)

    # Vẽ lại ảnh có box cho đẹp
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(vis_img)
    ax.axis("off")
    ax.set_title(f"Số món ăn phát hiện: {count}", fontsize=14)

    if verbose:
        print(f"Ảnh size: {pil_img.size}")
        print(f"Phát hiện {len(results)} vùng (bao gồm UNKNOWN).")
        print(f"Trong đó nhận diện được {count} món (!= 'UNKNOWN'):\n")
        for i, det in enumerate(detections, 1):
            box = tuple(int(v) for v in det["box"])
            print(
                f"[{i:2}] {det['label']:<20} | "
                f"cls={det['cls_conf']:.3f} | yolo={det['yolo_conf']:.3f} | box={box}"
            )

    return count, detections, fig, ax


In [ ]:
# Cell 4: Test đếm số món ăn trên 1 bàn

from pathlib import Path
from PIL import Image

TEST_IMG_PATH = Path("/home/mtl/Downloads/hhh.jpg")  # đổi lại path ảnh của bạn

if not TEST_IMG_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy ảnh: {TEST_IMG_PATH}")

img = Image.open(TEST_IMG_PATH).convert("RGB")
print("Ảnh size:", img.size)

count, detections, fig, ax = detect_and_count_foods(
    img,
    det_conf=0.1,   # có thể giảm xuống 0.3 nếu YOLO bỏ sót
    det_iou=0.5,
    cls_thres=0.3,  # giảm xuống 0.4 nếu EfficientNet khó tính quá
    enlarge=0.12,
    verbose=True,
)

plt.show()

print(f"\n✅ TỔNG SỐ MÓN ĂN PHÁT HIỆN: {count}")
